# 第三章：归一化、降维与聚类

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

上一篇 QC 之后，我们手里有 58,735 个高质量细胞。但此时数据还是一张 58,735 × 25,354 的巨大稀疏矩阵——你看不出任何结构。

这一篇要做的事情，就是把这张"看不见"的矩阵变成"看得懂"的图。具体来说，我们要经历四个关键步骤：

归一化：让不同测序深度的细胞具有可比性
高变基因选择：从 25,354 个基因中挑出最有信息量的 3,000 个
PCA 降维：从 3,000 维压缩到 30 维，去除噪声
UMAP + 聚类：在低维空间中把相似的细胞分成群体

每一步都有"为什么这么做"的深层逻辑。理解这些逻辑，比记住代码重要得多。

## Step 1：归一化——让细胞间具有可比性

### 问题：为什么不能直接比较原始 counts？

想象两个完全相同的细胞，一个被测序了 10,000 条 UMI，另一个只有 3,000 条。如果不做归一化，第一个细胞的所有基因表达量看起来都比第二个高 3 倍多——但这纯粹是技术噪声（采样深度不同），不是生物学差异。

归一化的目的就是消除这种"测序深度偏差"，让我们看到的差异尽可能反映真实的生物学变异。

### LogNormalize：简单有效的标准方法

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/01_after_qc.h5ad")
print(f"输入: {adata.n_obs} cells × {adata.n_vars} genes")

# 保留原始 counts（后面差异分析需要）
adata.layers["counts"] = adata.X.copy()

# Step 1: Library-size 归一化到 10,000
sc.pp.normalize_total(adata, target_sum=1e4)

# Step 2: 对数变换
sc.pp.log1p(adata)
print(f"归一化完成: X 范围 [{adata.X.min():.2f}, {adata.X.max():.2f}]")

**输出：**

> 📊 输出：
> 输入: 58735 cells × 25354 genes
> 归一化完成: X 范围 [0.00, 12.77]

两步操作的含义：

第一步 normalize_total(target_sum=1e4)：把每个细胞的 UMI 总数缩放到 10,000。为什么是 10,000？这是一个约定俗成的 scale factor，大约相当于中等深度测序的一个细胞的 UMI 总数。选 1,000 或 100,000 也可以，不影响下游分析——重要的是所有细胞用同一个数字。

第二步 log1p：log(x + 1) 变换。加 1 是为了处理 0 值（log(0) 未定义）。对数变换有两个关键作用：（1）将强右偏的 count 分布拉成近似正态；（2）压缩高表达基因的方差，让低表达基因也能被"看到"。

💡 为什么一定要保存原始 counts？

adata.layers["counts"] = adata.X.copy() 这行看起来不起眼，但至关重要。归一化后的数据适合降维和聚类，但差异表达分析和高变基因选择需要原始 counts。如果不保留，后续每次需要 counts 都要重新读取原始文件——而那是一个 1 GB 的文件。

## Step 2：高变基因选择——信号 vs 噪声

### 为什么不用全部 25,354 个基因

25,000 个基因中，大部分是"管家基因"（housekeeping genes）——在所有细胞类型中表达量差不多。这些基因对区分细胞类型没有帮助，反而引入噪声。另一些基因只在极少数细胞中检测到，统计信号太弱。

我们真正需要的是高变基因（Highly Variable Genes, HVGs）：在不同细胞之间表达变化最大的基因。这些基因最有可能携带区分不同细胞类型和状态的生物学信息。

### Seurat v3 方法选择 HVG

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    layer="counts",        # 用原始 counts 选 HVG
    batch_key="sample",    # 每个样本独立选，然后取交集
)
n_hvg = adata.var["highly_variable"].sum()
print(f"高变基因: {n_hvg}")

**输出：**

> 📊 输出：
> 高变基因: 3000

图 1：高变基因（HVG）选择——红色点为选中的 2000 个 HVG

这里有几个重要的参数决策：

n_top_genes=3000：选多少个 HVG？这是一个经验参数。太少（<1000）可能丢失重要信号，太多（>5000）引入噪声。2000-5000 是常用范围，3000 是一个安全的选择。如果你的数据细胞类型很多（>20 种），可以增加到 4000-5000。

flavor="seurat_v3"：使用 Seurat v3 的变异稳定化方法（VST），它对均值-方差关系建模，选出相对于其表达水平来说变异异常高的基因。这比简单的"最高方差"更合理——否则高表达基因会天然占优。

batch_key="sample"：这是关键——对每个样本独立选 HVG，然后取各样本 HVG 列表的交集。为什么？如果某个基因只在一个样本中高度变异（比如因为技术原因），它不应该被选为 HVG。跨样本一致的变异才是可靠的生物学信号。

⚠️ 踩坑预警：layer="counts" 不能省

Seurat v3 的 HVG 选择方法需要原始 counts，而不是归一化后的数据。如果你用归一化后的 adata.X，方差估计会不准确（因为 log 变换改变了方差结构）。这就是为什么第一步要保存 adata.layers["counts"]。

如果忘了保存 counts，你会看到一个 warning：flavor='seurat_v3' expects raw count data in the selected layer。很多人忽略这个 warning——不要忽略，它真的会影响 HVG 选择的质量。

## Step 3：PCA——从高维到低维

### 为什么需要 PCA

即使选了 3000 个 HVG，3000 维的数据仍然太多——不仅计算量大，而且存在严重的"维度灾难"：在高维空间中，距离度量变得不可靠，邻域图也会很嘈杂。

PCA（主成分分析）做的事情是：找到数据中方差最大的方向（主成分），把 3000 维数据投影到这些方向上，只保留前 N 个主成分。排名靠前的 PC 捕获了大部分生物学信号，排名靠后的 PC 主要是噪声。

In [ ]:
# 缩放（只对 HVG）——让每个基因贡献相同的方差
adata.raw = adata  # 保留全基因表达用于后续可视化
sc.pp.scale(adata, max_value=10)

# PCA（50 个主成分）
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")

# 查看方差解释
print(f"前 30 PC 解释方差: "
      f"{adata.uns['pca']['variance_ratio'][:30].sum()*100:.1f}%")

**输出：**

> 📊 输出：
> 前 30 PC 解释方差: 27.4%

adata.raw = adata：这行代码很容易被忽视，但非常重要。它保存了一份"未缩放、但已归一化"的完整基因表达矩阵。后续绘制基因表达的 UMAP（比如 sc.pl.umap(adata, color="CD68")）需要用到全部基因的归一化表达值，而不是缩放后的 3000 个 HVG。

sc.pp.scale(max_value=10)：缩放每个基因到均值为 0、方差为 1，并截断超过 10 的值。为什么要截断？因为有些基因在少数细胞中表达特别高，scale 后这些极端值可能主导 PCA 的结果。max_value=10 限制了这种影响。

### 选多少个 PC？

27.4% 的方差解释率——这数字看起来很低？不用担心，这在 scRNA-seq 中完全正常。原因是数据极度稀疏（93% 的矩阵是零），大量方差来自技术噪声和稀疏性。

关键不是总方差解释了多少，而是哪些 PC 携带生物学信号。我们画一个 elbow plot 来判断：

In [ ]:
import matplotlib.pyplot as plt

sc.pl.pca_variance_ratio(adata, n_pcs=50, show=False)
plt.savefig("results/figures/02_pca_variance.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 2：PCA 主成分方差解释比例（Elbow Plot）

图 1：PCA 方差解释率（elbow plot）。 前几个 PC 的方差贡献率急剧下降，到第 20-30 个 PC 后趋于平缓——这个"拐肘"就是信号和噪声的大致分界线。我们选择前 30 个 PC 用于下游分析。

💡 30 个 PC 够吗？

对于大多数数据集，20-50 个 PC 是合理的范围。选太少（<15）可能丢失细微的亚群信号，选太多（>50）则引入噪声。30 是一个安全的默认值。

有没有更客观的选择方法？有——比如 jackstraw 方法（Seurat 用的）或者 marchenko-pastur 随机矩阵理论法。但在实践中，这些方法的结论和 elbow plot 通常一致。花 1 小时跑 jackstraw 和花 10 秒看 elbow plot 得到的 PC 数差不了多少。

## Step 4：UMAP——看见数据的结构

### 从邻域图到 UMAP

UMAP 可能是 scRNA-seq 分析中最被"误用"的工具之一。先说清楚它能做什么和不能做什么：

✅ 展示细胞之间的局部相似性关系
✅ 帮助直观判断有多少个大致的细胞群体
❌ 群体之间的距离不代表它们之间的真实相似度
❌ 群体的大小不代表细胞的真实数量
❌ 群体的形状没有生物学意义

UMAP 先要构建邻域图——对每个细胞找到它在 PCA 空间中最近的 15 个邻居：

In [ ]:
# 构建 KNN 邻域图
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# 计算 UMAP
sc.tl.umap(adata, min_dist=0.3)
print("UMAP 完成")

**输出：**

> 📊 输出：
> UMAP 完成

n_neighbors=15：每个细胞连接几个最近邻？这个参数控制 UMAP 关注"局部"还是"全局"结构。小的 n_neighbors（如 5）强调局部细节，可能产生碎片化的图；大的 n_neighbors（如 50）强调全局结构，但细节会丢失。15 是一个平衡的默认值。

min_dist=0.3：UMAP 中点之间的最小距离。小的值（0.0-0.1）让群体更紧凑、分离更明显，但可能产生"伪分裂"；大的值（0.5+）让点更分散。0.3 是一个中等选择。

## Step 5：Leiden 聚类——把细胞分成群体

### 为什么用 Leiden 而不是 K-means

传统的 K-means 聚类有一个致命问题：你需要预先指定聚类数 K。在 scRNA-seq 中，你通常不知道数据里有多少种细胞类型。

Leiden 算法（Traag et al., 2019）是社区检测算法，基于邻域图工作。它不需要指定聚类数，而是通过 resolution 参数控制聚类的粒度：resolution 越大，聚类越多（每个群体越小）。

### 多 resolution 扫描

一个 resolution 通常不够。我们扫描一系列 resolution，看聚类数如何变化：

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
for res in resolutions:
    key = f"leiden_{res}"
    sc.tl.leiden(
        adata, resolution=res, key_added=key,
        flavor="igraph", n_iterations=2
    )
    n_clusters = adata.obs[key].nunique()
    print(f"  Leiden res={res}: {n_clusters} clusters")

# 默认用 res=0.8
adata.obs["leiden"] = adata.obs["leiden_0.8"]

**输出：**

> 📊 输出：
>   Leiden res=0.2: 15 clusters
>   Leiden res=0.4: 19 clusters
>   Leiden res=0.6: 22 clusters
>   Leiden res=0.8: 25 clusters
>   Leiden res=1.0: 28 clusters
>   Leiden res=1.2: 31 clusters
>   Leiden res=1.5: 37 clusters
>   Leiden res=2.0: 40 clusters

从 15 个到 40 个——怎么选？一般原则是：

先粗后细。 从低 resolution 开始（如 0.4-0.6），做完初步注释后，如果发现某个大 cluster 包含了不同的细胞亚型，再对该 cluster 用高 resolution 细分。
参考细胞类型数。 肝脏组织预期有 10-15 种主要细胞类型（免疫细胞、内皮细胞、肝细胞、星状细胞等）。加上亚型，20-30 个 cluster 是合理的。
观察稳定性。 如果从 res=0.8 到 res=1.0 新增的 3 个 cluster 都是从大 cluster 中分裂出来的小群，说明 0.8 的粗分已经足够。

我们暂时选 res=0.8（25 个 cluster） 作为起点。

⚠️ 踩坑预警：Leiden 的随机性

Leiden 聚类不是确定性的——每次运行结果可能略有不同。如果你需要可复现的结果，设置 random_state：sc.tl.leiden(adata, resolution=0.8, random_state=42)。

另外，flavor="igraph" vs flavor="leidenalg"：igraph 实现更快，但在某些边界情况下结果可能和 leidenalg 略有差异。对于大数据集（>50,000 细胞），igraph 是更好的选择。

## Step 6：可视化——第一眼印象

现在是最激动人心的时刻——看看 UMAP 长什么样：

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.umap(adata, color="condition", ax=axes[0], show=False,
           title="Condition", frameon=False)
sc.pl.umap(adata, color="leiden", ax=axes[1], show=False,
           title=f"Leiden (res=0.8, 25 clusters)",
           legend_loc="on data", legend_fontsize=8,
           frameon=False)
plt.tight_layout()
plt.savefig("results/figures/02_umap_condition_leiden.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 3：UMAP 按样本着色——可见明显的批次效应

图 2：UMAP 可视化。 左图按条件上色（蓝=Healthy，红=Cirrhotic），右图按 Leiden 聚类上色（25 个 cluster）。几个关键观察：

大部分 cluster 包含了两种条件的细胞——这是好事，说明疾病主要改变的是细胞比例和基因表达，而不是产生了全新的细胞类型。
有些区域明显偏向某一种条件——这可能反映了疾病特异的细胞亚群或状态变化。
UMAP 左侧和右侧的 cluster 分离很远——初步猜测这是免疫细胞（CD45+）和非免疫细胞（CD45-）的大分类。

再看看样本分布：

In [ ]:
# UMAP by sample（检查批次效应）
sc.pl.umap(adata, color="sample", show=False,
           title="Sample", frameon=False)
plt.savefig("results/figures/02_umap_sample.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 4：UMAP 按分选策略（CD45+/CD45-）着色

图 3：UMAP 按样本上色。 如果某些 cluster 几乎只包含某一个样本的细胞——那就是批次效应（batch effect），不是生物学差异。在下一篇中，我们将用 Harmony 处理这个问题。

图 5：UMAP 按疾病条件和 Leiden 聚类着色

💡 怎么判断需不需要做批次矫正？

看 UMAP 上色 by sample：如果同一种细胞类型的细胞按样本分成了不同的"团"，就需要矫正。一个定量的方法是计算 LISI（Local Inverse Simpson's Index）：高 LISI 说明邻域中样本混合良好，低 LISI 说明样本分离。

但要注意：不是所有样本分离都是"批次效应"。 如果 Healthy 和 Cirrhotic 样本在某些 cluster 中分开，这可能就是真实的疾病信号。判断的关键是：同一条件下的不同供体是否混合良好。

### 多 resolution 可视化

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(28, 12))
for i, res in enumerate(resolutions):
    ax = axes[i // 4, i % 4]
    key = f"leiden_{res}"
    n_cl = adata.obs[key].nunique()
    sc.pl.umap(adata, color=key, ax=ax, show=False,
               title=f"res={res} ({n_cl} clusters)",
               legend_loc="on data", legend_fontsize=6,
               frameon=False)
plt.tight_layout()
plt.savefig("results/figures/02_umap_multi_resolution.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 6：不同 Leiden 分辨率（0.2-2.0）的聚类效果对比

图 4：不同 resolution 下的 Leiden 聚类。 从 res=0.2（15 个 cluster）到 res=2.0（40 个 cluster），可以清晰地看到 cluster 的分裂过程。比如在低 resolution 下合为一体的免疫细胞大群，在高 resolution 下分裂成 T 细胞、NK 细胞、B 细胞等亚群。选择合适的 resolution 就像选择显微镜的倍数——取决于你想看多细的结构。

### 保存

In [ ]:
import os
out_path = "results/02_clustered.h5ad"
adata.write_h5ad(out_path)
print(f"已保存: {out_path}")
print(f"  文件大小: {os.path.getsize(out_path) / 1e6:.1f} MB")

**输出：**

> 📊 输出：
> 已保存: results/02_clustered.h5ad
>   文件大小: 1841.2 MB

## 与原文比较

📖 与 Ramachandran et al., 2019 对照

原文使用了 Seurat 进行归一化和聚类，而我们用的是 Scanpy。两者的核心步骤完全对应：

步骤
Seurat
Scanpy (本教程)

归一化
NormalizeData
normalize_total + log1p

HVG
FindVariableFeatures
highly_variable_genes

缩放
ScaleData
scale

PCA
RunPCA
pca

聚类
FindClusters (Louvain)
leiden

唯一的显著差异是聚类算法：原文用了 Louvain，我们用了 Leiden。Leiden 是 Louvain 的改进版，保证了聚类的连通性（Traag et al., 2019）。在实践中，两者的结果非常相似。

注意：此时我们还没有做批次矫正。 原文在聚类前做了 CCA 整合，相当于我们下一篇要做的 Harmony 整合。所以现在的聚类结果可能受到批次效应的影响——这是预期中的，也正是下一篇要解决的问题。

---

### 🔬 方法选择指南

🔬 方法选择指南：归一化、降维与聚类

归一化方法对比
方法原理优点局限推荐场景LogNormalize ✓总量归一 + log变换简单直观，广泛验证对零膨胀处理有限常规分析首选SCTransform正则化负二项回归更好处理技术噪声，整合方差稳定化与HVG选择计算较慢；Seurat生态大批次/技术差异大时scran池化反卷积估计size factor统计学更严谨需要预聚类；实现复杂追求统计严谨性时
我们的选择：LogNormalize（normalize_total + log1p）。这是 Scanpy/Seurat 默认方法，在大多数基准测试中表现稳健，且与下游所有工具兼容性最好。对于本数据集（5个样本、约10万细胞），LogNormalize 已足够。

降维可视化方法对比
方法原理保留结构速度推荐场景UMAP ✓流形学习 + 模糊拓扑局部+部分全局快首选可视化方法t-SNE概率分布匹配局部结构好中小数据集/局部细节展示ForceAtlas2力导向图布局依赖KNN图慢与PAGA结合使用PCA (直接)线性投影全局方差方向极快快速质检/无需非线性时
我们的选择：UMAP（min_dist=0.3）。UMAP 在速度和可视化质量之间取得了最佳平衡，且是目前单细胞领域的事实标准。注意：UMAP 坐标仅用于可视化，不应用于聚类或统计推断——聚类始终在 PCA 空间的 KNN 图上进行。

聚类算法对比
方法原理分辨率可调社区检测质量推荐Leiden ✓模块度优化 + 连通性保证是保证连通子图当前首选Louvain模块度优化（经典）是可能产生不连通社区老流程兼容K-means质心距离最小化需预设K不适合非球形仅作对照
为什么 Leiden 优于 Louvain：Louvain 算法的已知缺陷是可能产生内部不连通的社区（"poorly connected communities"）。Leiden 算法（Traag et al., 2019）在数学上保证了每个社区的连通性，且运行速度相当。目前 Scanpy 和 Seurat v5 均已默认使用 Leiden。

关于分辨率参数：没有"正确"的分辨率——它取决于生物学问题。本教程使用 res=0.8 得到25个cluster，但我们在注释章节（Ch05）会通过多分辨率扫描 + ARI/Silhouette 指标来系统性选择最优值。

---

## 方法论反思

### 归一化只是"凑合"

LogNormalize 是 scRNA-seq 领域最广泛使用的归一化方法，但它有已知的缺陷：它假设所有细胞的 RNA 总量相同（所谓的"composition assumption"），这在现实中不成立——肝细胞的 RNA 含量可能是 T 细胞的 10 倍。

更高级的方法（如 scran 的 pool-based normalization 或 SCTransform 的正则化负二项回归）可以部分解决这个问题。但在实践中，对于 QC 良好的数据，LogNormalize 和这些方法的下游结果差异很小。

### UMAP 不是真理

请记住这句话：UMAP 是一个可视化工具，不是分析工具。 所有基于 UMAP 图形做出的判断（"这两个群体离得很近所以它们相似"）都需要用定量方法验证。UMAP 的目的是帮助你生成假设，而不是证明假设。

## 小结

这一篇我们完成了：

✅ LogNormalize 归一化（normalize_total + log1p），保留原始 counts 到 layers
✅ Seurat v3 方法选择 3,000 个高变基因（batch_key="sample" 跨样本）
✅ PCA 降维到 50 个主成分（前 30 PC 解释 27.4% 方差）
✅ KNN 邻域图 + UMAP 可视化
✅ 多 resolution Leiden 聚类（0.2-2.0），选择 res=0.8（25 clusters）

关键数据流：

**输出：**

> 58,735 cells × 25,354 genes (log-normalized)
>   → 3,000 HVGs (seurat_v3, batch-aware)
>     → 50 PCs → 30 PCs (for neighbors)
>       → KNN graph (k=15) → UMAP
>         → 25 Leiden clusters (res=0.8)

下一篇，我们将用 Harmony 对 20 个文库的批次效应进行矫正。矫正前后的 UMAP 对比会让你直观地看到批次效应有多大——以及为什么多样本分析必须做这一步。